In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Hilangkan warning
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# Load dataset
df = pd.read_csv('../data/heart_disease_uci.csv')
print("Dataset berhasil dimuat!")
display(df.head())

In [ ]:
print("="*50)
print("1. MENELAAH DATA")
print("="*50)
# Menampilkan info seperti PDF Hal 5
df.info()

print("\n" + "="*50)
print("2. MEMVALIDASI DATA (Cek Inf & Null)")
print("="*50)
# Cek nilai Inf (seperti PDF Hal 6)
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
print(f"Jumlah nilai Inf pada kolom numerik: {np.isinf(df[num_cols]).sum().sum()}")

# Cek Null (seperti PDF Hal 7)
print("\nJumlah Missing Values (Null) per kolom:")
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
print("="*50)
print("3. MENENTUKAN OBJEK & LABEL")
print("="*50)

# Menampilkan 5 baris objek data (seperti PDF Hal 8)
print("Contoh Objek Data (5 Baris Pertama):")
display(df.head())

# Menentukan Label (Seperti PDF Hal 11, memetakan label)
# Target asli 'num' (0-4), kita ubah ke binary (0=Normal, 1=Sakit)
df['Category'] = df['num'].apply(lambda x: 'NORMAL' if x == 0 else 'SAKIT JANTUNG')

print("\nDistribusi Label Kategori:")
print(df['Category'].value_counts())

In [ ]:
print("="*50)
print("4. VISUALISASI DISTRIBUSI KATEGORI")
print("="*50)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Grafik 1: Distribusi Kategori (Setara PDF Hal 12)
sns.countplot(ax=axes[0], data=df, x='Category', order=df['Category'].value_counts().index, palette='viridis')
axes[0].set_title('Distribusi Kategori Heart Disease (Setara PDF Hal 12)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Category')
axes[0].set_ylabel('Jumlah Record')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

# Grafik 2: Binary Classification (Setara PDF Hal 13 & 14)
# Karena dataset kita hanya punya 2 kelas (Normal & Sakit), grafiknya sama saja, tapi kita buat lebih tegas
sns.countplot(ax=axes[1], data=df, x='Category', palette=['#2ca02c', '#d62728'])
axes[1].set_title('Binary Classification (Setara PDF Hal 13 & 14)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Jumlah Record')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()), 
                     ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print("="*50)
print("5. MEMBERSIHKAN DATA")
print("="*50)

# 1. Konversi Inf ke NaN (Seperti PDF Hal 17)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Imputasi Nilai Kosong
# Untuk numerik pakai median (Seperti PDF Hal 19)
for col in ['trestbps', 'chol', 'thalch', 'oldpeak', 'ca']:
    df[col] = df[col].fillna(df[col].median())

# Untuk kategorikal pakai modus
for col in ['fbs', 'restecg', 'exang', 'slope', 'thal', 'cp']:
    df[col] = df[col].fillna(df[col].mode()[0])

# Cek ulang null
print(f"Sisa Missing Values setelah imputasi: {df.isnull().sum().sum()}")

# 3. Cek & Hapus Duplikat (Seperti PDF Hal 20-21)
duplikat = df.duplicated().sum()
print(f"Total baris duplikat ditemukan: {duplikat}")
df.drop_duplicates(inplace=True)
print(f"Data setelah dihapus duplikat: {df.shape}")

In [ ]:
print("="*50)
print("6. MENGKONSTRUKSI DATA")
print("="*50)

# 1. Encoding (Seperti PDF Hal 24)
# Konversi boolean/string ke angka dulu agar bisa di-encode
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})
df['fbs'] = df['fbs'].map({'TRUE': 1, 'FALSE': 0})
df['exang'] = df['exang'].map({'TRUE': 1, 'FALSE': 0})

# One Hot Encoding
df_encoded = pd.get_dummies(df, columns=['cp', 'restecg', 'slope', 'thal'], drop_first=True, dtype=int)
print(f"Shape setelah encoding: {df_encoded.shape}")

# 2. Label Target biner (Seperti PDF Hal 25)
df_encoded['Target'] = (df_encoded['num'] > 0).astype(int)

# 3. Drop kolom yang tidak dipakai & Tentukan X, y (Seperti PDF Hal 26)
X = df_encoded.drop(columns=['id', 'dataset', 'num', 'Category', 'Target'])
y = df_encoded['Target']

# 4. Seleksi Fitur via Korelasi (Seperti PDF Hal 27-28)
korelasi_target = X.corrwith(y).abs().sort_values(ascending=False)
print("\nNilai Korelasi Fitur terhadap Target:")
display(korelasi_target.head(10)) # Tampilkan 10 teratas

# Ambil fitur dengan korelasi >= 0.20 (Seperti PDF Hal 28)
fitur_penting = korelasi_target[korelasi_target >= 0.20].index.tolist()
X_baru = X[fitur_penting]

print(f"\nJumlah fitur dipakai: {len(fitur_penting)}")
print(f"Fitur terpilih: {fitur_penting}")
display(X_baru.head())

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

print("="*50)
print("7. MEMBANGUN MODEL")
print("="*50)

# 1. Split Data 80/20 (Seperti PDF Hal 29)
X_train, X_test, y_train, y_test = train_test_split(
    X_baru, y, test_size=0.2, random_state=42, stratify=y
)
print("Data berhasil dibagi menjadi set Training dan Testing.")

# 2. Standarisasi (Seperti PDF Hal 30)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Training Logistic Regression (Seperti PDF Hal 31)
log_reg = LogisticRegression(
    class_weight='balanced', solver='liblinear', random_state=42, max_iter=1000
)
log_reg.fit(X_train_scaled, y_train)
print("Pelatihan model Regresi Logistik selesai. Model siap dievaluasi.")

In [ ]:
print("="*50)
print("8. EVALUASI MODEL")
print("="*50)

# Prediksi
y_pred = log_reg.predict(X_test_scaled)

# 1. Classification Report (Seperti PDF Hal 32)
print("### Classification Report ###")
print(classification_report(y_test, y_pred, target_names=['NORMAL (0)', 'SAKIT (1)']))

# 2. Confusion Matrix (Seperti PDF Hal 33)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['NORMAL (0)', 'SAKIT (1)'], 
            yticklabels=['NORMAL (0)', 'SAKIT (1)'])
plt.title('Confusion Matrix')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

In [ ]:
import joblib

print("="*50)
print("9. EXPORT MODEL")
print("="*50)

joblib.dump(log_reg, "../models/logreg_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(fitur_penting, "../models/features.pkl")

print("Model berhasil di-export:")
print("- logreg_model.pkl")
print("- scaler.pkl")
print("- features.pkl")